In [21]:
# Create quarterly revenue and target data by region and salesperson
import pandas as pd
sales = pd.DataFrame({
    "region": ["North", "North", "North", "South", "South", "West"],
    "salesperson": ["A", "A", "B", "C", "C", "D"],
    "quarter": ["Q1", "Q2", "Q1", "Q1", "Q2", "Q1"],
    "revenue": [1000, 1200, 900, 1500, 1900, 800],
    "target": [1100, 1100, 1000, 1400, 1400, 900],
})
sales

,region,salesperson,quarter,revenue,target
0,North,A,Q1,1000,1100
1,North,A,Q2,1200,1100
2,North,B,Q1,900,1000
3,South,C,Q1,1500,1400
4,South,C,Q2,1900,1400
5,West,D,Q1,800,900


In [22]:
# Aggregate revenue and targets at regional level
regional_totals=sales.groupby('region',as_index=False)[["revenue","target"]].sum()
regional_totals

,region,revenue,target
0,North,3100,3200
1,South,3400,2800
2,West,800,900


In [23]:
# Calculate each quarter's percentage of target achieved
sales["achievement_%"]=round(sales["revenue"].div(sales["target"])*100,2)
sales

,region,salesperson,quarter,revenue,target,achievement_%
0,North,A,Q1,1000,1100,90.91
1,North,A,Q2,1200,1100,109.09
2,North,B,Q1,900,1000,90.00
3,South,C,Q1,1500,1400,107.14
4,South,C,Q2,1900,1400,135.71
5,West,D,Q1,800,900,88.89


In [24]:
# Summarize revenue for each salesperson within each region
salesperson_summary=(
    sales.groupby(["region","salesperson"])["revenue"]
    .sum()
    .reset_index()
)
salesperson_summary

,region,salesperson,revenue
0,North,A,2200
1,North,B,900
2,South,C,3400
3,West,D,800


In [25]:
# Rank salespeople independently within each region
salesperson_summary["regional_rank"]=(
    salesperson_summary.groupby('region')["revenue"]
    .rank(method="dense",ascending=False)
    .astype(int)
)
salesperson_summary

,region,salesperson,revenue,regional_rank
0,North,A,2200,1
1,North,B,900,2
2,South,C,3400,1
3,West,D,800,1


In [26]:
# Calculate each salesperson's share of regional revenue
salesperson_summary["regional_revenue_contribution"]=(
    (salesperson_summary["revenue"]/salesperson_summary.groupby('region')["revenue"].transform("sum"))*100
).round(2)
salesperson_summary

,region,salesperson,revenue,regional_rank,regional_revenue_contribution
0,North,A,2200,1,70.97
1,North,B,900,2,29.03
2,South,C,3400,1,100.00
3,West,D,800,1,100.00


In [27]:
# Flag quarters where regional performance is below target
sales["region_below_target"]=sales["achievement_%"]<100
sales

,region,salesperson,quarter,revenue,target,achievement_%,region_below_target
0,North,A,Q1,1000,1100,90.91,True
1,North,A,Q2,1200,1100,109.09,False
2,North,B,Q1,900,1000,90.00,True
3,South,C,Q1,1500,1400,107.14,False
4,South,C,Q2,1900,1400,135.71,False
5,West,D,Q1,800,900,88.89,True


In [32]:
# Find Q1 salespeople who met or exceeded their target
# Salesperson who exceeded target in at least one quarter
extra_value_salesperson=sales[(sales["revenue"]>=sales["target"]) & (sales["quarter"]=="Q1")]
extra_value_salesperson

,region,salesperson,quarter,revenue,target,achievement_%,region_below_target
3,South,C,Q1,1500,1400,107.14,False


### Important notes

- Calculate achievement from aggregated revenue and target when reporting regional or salesperson-level performance; row-level percentages should not be averaged blindly.
- `groupby().rank()` ranks within each region, while `transform("sum")` broadcasts regional totals back to salesperson rows.
- Dense ranking gives tied salespeople the same rank without gaps; choose a tie-breaker when a unique ordering is required.
- Check for zero or missing targets before dividing because they can produce invalid achievement percentages.
- “At least one quarter” requires grouping or deduplicating salesperson results if a salesperson can satisfy the condition multiple times.
- Compare periods with consistent target definitions and account for partial quarters or missing sales records.
